# Checkpoint B solutions -- the Part-2 recommender

These reference solutions close **Part 2**. Across eight questions we score each Part-2 path on
`val` (semantic retrieval, the two-tower, the sparse + dense hybrid), read the cold-start, sequence
and ethics readouts, build the pinned Part-2 blend, ablate it, FREEZE it, and finally give the frozen
blend one honest grade on the sealed `test` split of the main log.

The discipline is the same as Checkpoint A: **freeze, then score**. Questions 1-7 run on `val` only
(Question 5 on the *session* log's `val`). The `# FREEZE` cell at the end of Question 7 settles every
decision; Question 8 is the **only** cell that names the `test` split, in exactly one top-level call,
and it never returns to selection.

Everything is seeded (`SEED = 0`), measured at `K = 10` with the 60 cold readers excluded, and each
question ends with an `assert` on a computed result. The honest thesis, stated up front: the pinned
Part-2 blend is a **hit <-> coverage trade, not a win** -- it loses on hit@10 to its own two-tower
member and to the hybrid, and buys about twice their catalog reach.

## Setup -- the data and two cheap Part-1 paths

The two cheap Part-1 paths are given; the setup also builds the train-only seen sets, the warm
population, the zero-train item set and each reader's `val` positives. `test` stays SEALED.

In [1]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LastKItemItemPath,
    LexicalRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TransitionRetrievalPath,
    TwoTowerRetrievalPath,
    blend,
    book_embedding,
    exposure_gini,
    genre_spread_vs_history,
    head_ids_from_counts,
    load_glove_subset,
    load_train_sequences,
    long_tail_share,
    rank,
    reader_hits,
    run_blended_scoreboard,
    run_validation_scoreboard,
    simulate_feedback_loop,
    val_cohort,
)
from bookrec.lexical import tokenize

SEED = 0
K = 10      # every metric is reported at the top-10 cutoff
POOL = 30   # the per-path retrieval pool for blends (Unit 6)

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)        # the 2000-book catalog: the coverage / Gini denominator
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])   # 60 readers no personalized path can serve

rows = pd.read_csv(interactions_path).to_dict("records")

# Train positives are each reader's `seen` set; `val` positives are the held-out relevant set.
seen_by = defaultdict(set)
val_by = defaultdict(set)
train_counts = defaultdict(int)      # book -> number of train positives
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
        train_counts[row["item_id"]] += 1
    elif row["split"] == "val":
        val_by[row["reader_id"]].add(row["item_id"])

# Every non-cold reader with a train history: the label-free population for exposure readouts.
warm_population = sorted(reader for reader in seen_by if reader not in cold_readers)
# Books with NO train positive: the cold-item denominator for Question 4.
zero_train_items = set(catalog_ids) - set(train_counts)

item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)


print(f"books: {len(catalog_ids)}   warm population: {len(warm_population)}   "
      f"zero-train items: {len(zero_train_items)}   fitted: {[item_item.name, lexical.name]}")

assert len(catalog_ids) == 2000, "the catalog is the 2000-book shelf"
assert not (set(warm_population) & cold_readers), "the warm population excludes cold readers"
assert zero_train_items, "some catalog books have no train positive (the cold items)"

books: 2000   warm population: 540   zero-train items: 818   fitted: ['item-item', 'lexical']


## Question 1 -- Meaning, not words: semantic retrieval

Lexical retrieval (Unit 3) matches a reader's history to books that share *words*. Semantic retrieval
(Unit 7) matches *meaning*: every keyword becomes a pretrained **word embedding** (the committed GloVe
subset), a book becomes the L2-normalized mean of its keyword vectors (a **document embedding**), a
reader becomes the mean of the books they read, and retrieval returns the books with the highest
cosine (**embedding retrieval**).

1. Build one book's embedding by hand with `book_embedding(tokenize(keywords[book]), glove)` for
   `book = catalog_ids[0]`, and record its norm (it should be 1, up to rounding).
2. Fit a `SemanticEmbeddingRetrievalPath` and score it on `val` with `run_validation_scoreboard`
   (measured on this data: hit@10 about 0.102, *below* lexical's 0.158).

Then, in two or three sentences: why can the semantic path score below lexical on hit@10 and still be
worth keeping in a system? (Think about what it surfaces that lexical does not.)

In [2]:
glove = load_glove_subset()

sample_book = catalog_ids[0]
sample_embedding = book_embedding(tokenize(keywords[sample_book]), glove)
sample_norm = float(np.linalg.norm(sample_embedding))   # a document embedding is L2-normalized

semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
semantic_val = run_validation_scoreboard(
    semantic, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers,
)                                      # split defaults to "val"
semantic_hit = semantic_val.hit_rate_at_k

lexical_val = run_validation_scoreboard(
    lexical, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers,
)
lexical_hit = lexical_val.hit_rate_at_k

semantic_reading = (
    "Semantic retrieval matches MEANING, so it reaches books whose keywords are different words "
    "from the reader's history (a 'dragon' reader can be matched to a 'wyvern' book), and those "
    "candidates overlap little with lexical's. On its own it scores below lexical here "
    "(hit@10 0.102 vs 0.158), but a weaker path that finds DIFFERENT right answers can still "
    "add reach and complementary candidates to a blend -- which is exactly how Unit 7 measured a "
    "blend lift from it."
)

print(f"sample book {sample_book}: embedding dim {sample_embedding.shape[0]}, norm {sample_norm:.6f}")
print(f"val hit@10   semantic {semantic_hit:.3f}   lexical {lexical_hit:.3f}   "
      f"({semantic_val.readers} readers)")

assert abs(sample_norm - 1.0) < 1e-6, "a book embedding is L2-normalized"
assert 0.08 < semantic_hit < 0.13, "semantic val hit@10 is about 0.102"
assert 0.13 < lexical_hit < 0.19, "lexical val hit@10 is about 0.158"
assert semantic_hit < lexical_hit, "on this data semantic scores BELOW lexical on hit@10"

sample book 0: embedding dim 100, norm 1.000000
val hit@10   semantic 0.102   lexical 0.158   (500 readers)


## Question 2 -- A learned model: the two-tower

The two-tower (Unit 8) learns a **reader tower** and a **book tower** in PyTorch: a reader's score for
a book is the dot product of their two embeddings. It is trained with the **BPR loss** -- for a reader,
a book they read should outscore a sampled book they did not -- by mini-batch gradient descent with
weight decay (**neural training**).

Fit **one** `TwoTowerRetrievalPath(seed=SEED)` on the main log (about 15-25 s; fit it once and reuse
it for the rest of the checkpoint). Record the first and last epoch of its BPR loss history, and score
it on `val`. Then answer in two or three sentences: what does a *falling* BPR loss tell you, and what
does it *not* tell you about hit@10 on held-out readers?

In [3]:
# The ONE two-tower fit of the checkpoint (~15-25 s); every later question reuses it.
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)

first_loss = float(two_tower.loss_history[0])
last_loss = float(two_tower.loss_history[-1])
loss_fell = bool(last_loss < first_loss)

two_tower_val = run_validation_scoreboard(
    two_tower, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers,
)
two_tower_hit = two_tower_val.hit_rate_at_k

bpr_reading = (
    "A falling BPR loss (0.370 at epoch 1 -> 0.111 at epoch 60) says the model got better at ranking each reader's "
    "TRAIN positives above sampled books they did not read -- it is a statement about the data "
    "the model was fit on. It says nothing directly about books the model has never seen as "
    "positives: a model can drive training loss down by memorizing train pairs while held-out "
    "accuracy stalls or falls. Only the held-out scoreboard measures that, and here it reads "
    "hit@10 0.340 on val, the best single path."
)

print(f"BPR loss: epoch 1 {first_loss:.4f} -> epoch {len(two_tower.loss_history)} {last_loss:.4f}"
      f"   (fell: {loss_fell})")
print(f"two-tower val hit@10 {two_tower_hit:.3f}")

assert loss_fell, "BPR training lowers the training loss"
assert 0.30 < two_tower_hit < 0.38, "the two-tower val hit@10 is about 0.340"
assert two_tower_hit > lexical_hit > semantic_hit

BPR loss: epoch 1 0.3700 -> epoch 60 0.1109   (fell: True)
two-tower val hit@10 0.340


## Question 3 -- Sparse + dense: the hybrid

Unit 10's **hybrid retrieval** fuses a sparse path (BM25 / lexical) and a dense path (the two-tower):
each path retrieves a pool of 50 candidates, their scores are min-max normalized, and a weighted sum
(dense weight 0.7) orders the union. Build it from the paths you already fit -- **no new fit** -- with
`HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50).fit(rows)`.

Measure hit@10 and catalog coverage for the two-tower alone and for the hybrid, each as a single-path
blend through `run_blended_scoreboard` with `pool=K` (measured: two-tower about 0.340 / 0.177, hybrid
about 0.362 / 0.192). In two or three sentences: what did fusion buy here, and why is "fusion is a free
win" still the wrong lesson? (Recall what equal weights did in Unit 10.)

In [4]:
def single_path_board(path):
    # A single-path blend is just that path; the blended scoreboard also reports coverage.
    return run_blended_scoreboard(
        {path.name: path}, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=K, cold_readers=cold_readers,
    )


# Built from the paths already fit (BM25 + the two-tower) -- no new model fit.
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50).fit(rows)

two_tower_board = single_path_board(two_tower)
hybrid_board = single_path_board(hybrid)

two_tower_coverage = two_tower_board.catalog_coverage
hybrid_hit = hybrid_board.hit_rate_at_k
hybrid_coverage = hybrid_board.catalog_coverage

hybrid_reading = (
    "At this tuned, dense-heavy weight (0.7 on the two-tower) fusion bought a small lift on BOTH "
    "axes: hit@10 0.340 -> 0.362 and coverage 0.177 -> 0.192, because BM25 contributes "
    "keyword matches the two-tower ranks low. But the lift is weight-sensitive -- in Unit 10 the "
    "equal-weight hybrid LOST to the two-tower alone -- so 'fusion is a free win' is the wrong "
    "lesson: a fusion weight is a choice that has to be measured on val, not assumed."
)

print(f"{'path':10s} {'hit@10':>7s} {'coverage':>9s}")
print(f"{'two-tower':10s} {two_tower_board.hit_rate_at_k:7.3f} {two_tower_coverage:9.3f}")
print(f"{'hybrid':10s} {hybrid_hit:7.3f} {hybrid_coverage:9.3f}")

assert abs(two_tower_board.hit_rate_at_k - two_tower_hit) < 1e-12, "same path, same cohort, same hit"
assert hybrid_hit > two_tower_hit, "the tuned hybrid lifts hit@10 a little"
assert hybrid_coverage > two_tower_coverage, "...and coverage a little"
assert hybrid_hit - two_tower_hit < 0.05, "the lift is small, not a leap"

path        hit@10  coverage
two-tower    0.340     0.177
hybrid       0.362     0.192


## Question 4 -- Cold start: who can recommend a book nobody has read?

A **cold item** is a book with no train positive. The setup cell built `zero_train_items`, the set of
catalog books no reader has a train positive for. Using **only paths you have already fit** -- semantic,
lexical, item-item and the two-tower; no new model -- measure two things per path:

1. **Cold reach:** over every reader in `warm_population`, retrieve that reader's top-10 (excluding
   their seen set) and count the *distinct* zero-train books that appear anywhere. This needs no
   held-out labels.
2. **Cold hit:** over the non-cold readers who have at least one zero-train book among their unseen
   `val` positives, the fraction whose top-10 contains one of those cold books.

Then, in two or three sentences: which kinds of path serve cold books and which surface exactly
zero, and *why* -- what does each kind of path need to know about a book before it can recommend it?
(Unit 9's feature tower is the learned path built to close this gap; you do not need to fit it here.)

In [5]:
def cold_reach(path):
    # Distinct zero-train books in any warm reader's top-10 (train history only, no labels).
    surfaced = set()
    for reader in warm_population:
        seen = seen_by[reader]
        top = rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)
        surfaced |= {candidate.item_id for candidate in top} & zero_train_items
    return len(surfaced)


def cold_hit(path):
    # hit@10 restricted to readers whose unseen val positives include a zero-train book.
    hits = []
    for reader in sorted(val_by):
        if reader in cold_readers:
            continue
        seen = seen_by[reader]
        cold_targets = (val_by[reader] - seen) & zero_train_items
        if not cold_targets:
            continue
        top = rank(path.retrieve(reader, {"seen": seen}, K), n=K, exclude=seen)
        hits.append(1.0 if {candidate.item_id for candidate in top} & cold_targets else 0.0)
    return (sum(hits) / len(hits) if hits else 0.0), len(hits)


cold_rows = []
for name, path in (("semantic", semantic), ("lexical", lexical), ("item-item", item_item),
                   ("two-tower", two_tower)):
    reach = cold_reach(path)
    hit, n_readers = cold_hit(path)
    cold_rows.append({"path": name, "zero_train_surfaced": reach, "cold_hit@10": hit,
                      "cold_target_readers": n_readers})

cold_by = {row["path"]: row for row in cold_rows}

cold_reading = (
    "The content paths place a book from its KEYWORDS, which every book has whether or not anyone "
    "has read it, so they DO surface cold books (semantic 268, lexical 128 distinct zero-train "
    "books) and occasionally hit a cold target (cold hit@10 0.031 and 0.021 over 97 readers). The warm-only collaborative paths (item-item, "
    "two-tower) learn a book only from its interactions; a book with no train positive has no "
    "co-reads and no trained embedding signal, so they surface exactly 0 cold books and score 0 "
    "cold hit. A path needs SOMETHING about a book -- content or interactions -- before it can "
    "recommend it; Unit 9's feature tower is the learned path that gets content into the model."
)

print(f"{'path':10s} {'cold reach':>10s} {'cold hit@10':>12s} {'readers':>8s}")
for row in cold_rows:
    print(f"{row['path']:10s} {row['zero_train_surfaced']:10d} {row['cold_hit@10']:12.3f} "
          f"{row['cold_target_readers']:8d}")

assert cold_by["semantic"]["zero_train_surfaced"] > 0 and cold_by["lexical"]["zero_train_surfaced"] > 0
assert cold_by["item-item"]["zero_train_surfaced"] == 0, "item-item cannot surface a never-read book"
assert cold_by["two-tower"]["zero_train_surfaced"] == 0, "the warm-only two-tower surfaces 0 cold books"
assert cold_by["item-item"]["cold_hit@10"] == 0.0 and cold_by["two-tower"]["cold_hit@10"] == 0.0
assert len({row["cold_target_readers"] for row in cold_rows}) == 1, "one shared cold-target cohort"

path       cold reach  cold hit@10  readers
semantic          268        0.031       97
lexical           128        0.021       97
item-item           0        0.000       97
two-tower           0        0.000       97


## Question 5 -- Order matters: sequence-aware paths (session log, `val` only)

Unit 12 used a **different log**: `sessions.csv.gz`, whose readers' reads come in a meaningful order
(series volumes, sessions). Everything in this question runs on that session log and on its `val`
split only; its numbers are **not comparable** with the main-log numbers elsewhere in this checkpoint.

Fit four cheap numpy paths on the session log:
- `ItemItemRetrievalPath()` -- the whole-history "bag" baseline;
- `LastKItemItemPath(3)` -- item-item queried with only the reader's last 3 distinct reads (recency);
- `TransitionRetrievalPath(5)` -- first-order item-to-item transitions from the last reads;
- `TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED, name="transition-shuffled")` -- the same
  path trained on each reader's history in a **shuffled order**, the control that destroys order but
  keeps every book.

Score each with `reader_hits` (per-reader hit@10 on the scoreboard's cohort; no `cold_readers` on the
session log) and compute two **paired** differences with their paired standard error
(`diffs.std(ddof=1) / sqrt(n)`): last-3 minus bag, and transition minus its shuffled control. Then, in
two or three sentences: what does each comparison show, and why is the shuffled control a stronger
piece of evidence for *order* than simply "the transition path scores well"?

In [6]:
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control seed (committed data seed + 1000)

sessions_path = data_dir / "sessions.csv.gz"           # the SESSION log -- different readers
session_rows = pd.read_csv(sessions_path).to_dict("records")
histories = load_train_sequences(sessions_path)        # each reader's ORDERED train positives

bag = ItemItemRetrievalPath().fit(session_rows, catalog=catalog_ids)
last_3 = LastKItemItemPath(3).fit(session_rows, catalog=catalog_ids)
transition = TransitionRetrievalPath(5).fit(session_rows, catalog=catalog_ids)
transition_shuffled = TransitionRetrievalPath(
    5, shuffle_seed=SHUFFLE_SEED, name="transition-shuffled",
).fit(session_rows, catalog=catalog_ids)

session_hits = {}
for label, path in (("bag", bag), ("last-3", last_3), ("transition", transition),
                    ("transition-shuffled", transition_shuffled)):
    session_hits[label] = reader_hits(path, sessions_path, catalog_ids=catalog_ids, k=K)


def paired(label_a, label_b):
    # Mean per-reader hit difference (a - b) and its paired standard error.
    readers = sorted(session_hits[label_a])
    diffs = np.array([session_hits[label_a][r] - session_hits[label_b][r] for r in readers])
    return float(diffs.mean()), float(diffs.std(ddof=1) / np.sqrt(len(diffs)))


recency_lift, recency_se = paired("last-3", "bag")
order_gap, order_se = paired("transition", "transition-shuffled")
session_mean = {label: float(np.mean(list(h.values()))) for label, h in session_hits.items()}

sequence_reading = (
    "On the session log, querying item-item with only the last 3 reads beats the whole-history "
    "bag by +0.071 hit@10 (paired SE 0.014; 0.551 vs 0.480): recent reads predict the next one better than the whole "
    "history. The transition path beats its OWN shuffled copy by +0.183 (paired SE 0.014; 0.516 vs 0.333). That "
    "control is stronger evidence for ORDER than 'the transition path scores well', because the "
    "shuffled copy keeps every book, the same model and the same readers and destroys only the "
    "order -- so the whole gap is attributable to order itself, not to which books were read."
)

n_session_readers = len(session_hits["bag"])
print(f"session-log val cohort: {n_session_readers} readers")
for label, mean_hit in session_mean.items():
    print(f"  {label:20s} hit@10 {mean_hit:.3f}")
print(f"last-3 - bag:                    {recency_lift:+.3f}  (paired SE {recency_se:.3f})")
print(f"transition - transition-shuffled: {order_gap:+.3f}  (paired SE {order_se:.3f})")

assert all(set(h) == set(session_hits["bag"]) for h in session_hits.values()), "one paired cohort"
assert recency_lift > 2 * recency_se, "recency beats the bag by more than two paired SEs"
assert order_gap > 2 * order_se, "order carries signal beyond the shuffled control"

session-log val cohort: 1419 readers
  bag                  hit@10 0.480
  last-3               hit@10 0.551
  transition           hit@10 0.516
  transition-shuffled  hit@10 0.333
last-3 - bag:                    +0.071  (paired SE 0.014)
transition - transition-shuffled: +0.183  (paired SE 0.014)


## Question 6 -- Ethics: exposure, filter bubbles, and a feedback loop

Unit 13 asked who a recommender is good for and what it does to its own data. Answer three parts.
Every claim you write must come with one of your numbers.

**The Part-2 blend configuration** for this checkpoint is pinned: an **equal-weight** blend of
**item-item, two-tower, semantic and lexical**, each contributing a pool of `POOL = 30` candidates.
(You measure it on `val` and freeze it in Question 7; here you only look at what it *serves*.)

**(a) Exposure fairness.** For every reader in `warm_population`, compute the blend's top-10 with
`blend` + `rank` (the helper below), and the two-tower's own top-10 for comparison. For both, report
the **exposure Gini** over the full catalog (`exposure_gini`) and the **long-tail share** (the
fraction of slots that go to non-head books; the head is the top 10% of books by train count,
`head_ids_from_counts(train_counts, catalog_ids)`). These readouts use no held-out labels.

**(b) Filter bubbles.** With `genre_spread_vs_history(blend_lists, seen_by, catalog)`, compare each
reader's recommended genre entropy with the genre entropy of their own history. Report the mean of
each, and the fraction of readers whose recommendations are *narrower* than their history.

**(c) A feedback loop, with and without exploration.** Run `simulate_feedback_loop` twice on
item-item CF, for `rounds=3` (short, to keep the checkpoint fast) over the val cohort
(`val_cohort(rows, catalog_ids, cold_readers)`), `seed=SEED`, once with no mitigation and once with
`mitigation="explore"` (one random unseen book in the last slot). Pass `path_factory` as a function
that returns a *fresh, unfitted* `ItemItemRetrievalPath()` -- the loop refits it every round. Compare
round 0 with round 3 on coverage, Gini, genre entropy and clean hit.

Then, in four to six sentences: what do (a)-(c) show; why is no single number here "the fairness" of
the system; and why must the loop never let a simulated reader "accept" one of their own `val` (or
other held-out) books -- what would that do to the clean-hit column?

In [7]:
PART2_PATHS = {"item-item": item_item, "two-tower": two_tower,
               "semantic": semantic, "lexical": lexical}
PART2_WEIGHTS = {name: 1.0 for name in PART2_PATHS}   # equal weights


def blend_top_k(paths, weights, reader):
    # The blend's top-10 for one reader: per-path pools -> calibrated weighted union -> rank.
    seen = seen_by[reader]
    per_path = {name: path.retrieve(reader, {"seen": seen}, POOL) for name, path in paths.items()}
    return rank(blend(per_path, weights=weights), n=K, exclude=seen)


head_ids = head_ids_from_counts(train_counts, catalog_ids)

# (a) exposure: what each recommender SERVES to the warm population (no labels needed)
blend_lists = {reader: blend_top_k(PART2_PATHS, PART2_WEIGHTS, reader)
               for reader in warm_population}
two_tower_lists = {
    reader: rank(two_tower.retrieve(reader, {"seen": seen_by[reader]}, K),
                 n=K, exclude=seen_by[reader])
    for reader in warm_population
}

blend_gini = exposure_gini(blend_lists, catalog_ids)
two_tower_gini = exposure_gini(two_tower_lists, catalog_ids)
blend_long_tail = long_tail_share(blend_lists, head_ids)
two_tower_long_tail = long_tail_share(two_tower_lists, head_ids)

# (b) filter bubbles: recommended genre spread vs each reader's own history
spread = genre_spread_vs_history(blend_lists, seen_by, catalog)
mean_rec_entropy = float(np.mean([rec for rec, _ in spread.values()]))
mean_history_entropy = float(np.mean([hist for _, hist in spread.values()]))
narrower_fraction = float(np.mean([rec < hist for rec, hist in spread.values()]))

print(f"head set: {len(head_ids)} books (top 10% by train count)")
print(f"{'recommender':10s} {'exposure Gini':>14s} {'long-tail share':>16s}")
print(f"{'blend':10s} {blend_gini:14.3f} {blend_long_tail:16.3f}")
print(f"{'two-tower':10s} {two_tower_gini:14.3f} {two_tower_long_tail:16.3f}")
print(f"genre entropy: recommended {mean_rec_entropy:.3f} bits vs history "
      f"{mean_history_entropy:.3f} bits; narrower than history for {narrower_fraction:.1%} of readers")

assert len(head_ids) == 200, "the head is the top 10% of the 2000-book catalog"
assert all(len(top) == K for top in blend_lists.values()), "every warm reader gets a full top-10"
assert blend_gini < two_tower_gini, "the blend spreads exposure more evenly than the two-tower"
assert 0.0 <= narrower_fraction <= 1.0 and len(spread) == len(warm_population)

head set: 200 books (top 10% by train count)
recommender  exposure Gini  long-tail share
blend               0.857            0.429
two-tower           0.929            0.100
genre entropy: recommended 0.811 bits vs history 1.551 bits; narrower than history for 76.3% of readers


In [8]:
# (c) two short feedback loops (R = 3) on item-item CF -- each loop runs ONCE.
cohort = val_cohort(rows, catalog_ids, cold_readers)


def fresh_item_item():
    return ItemItemRetrievalPath()   # a fresh, UNFITTED path -- the loop calls fit itself


loop_plain = simulate_feedback_loop(fresh_item_item, rows, cohort=cohort,
                                    catalog=catalog, seed=SEED, rounds=3)
loop_explore = simulate_feedback_loop(fresh_item_item, rows, cohort=cohort,
                                      catalog=catalog, seed=SEED, rounds=3, mitigation="explore")

loop_rows = []
for label, loop in (("no mitigation", loop_plain), ("exploration slot", loop_explore)):
    for metrics in (loop[0], loop[-1]):   # round 0 and round 3
        loop_rows.append({
            "loop": label,
            "round": metrics.round,
            "coverage": metrics.coverage,
            "gini": metrics.gini,
            "genre_entropy": metrics.genre_entropy,
            "clean_hit": metrics.clean_hit,
        })

ethics_reading = (
    "(a) The blend spreads exposure more evenly than its two-tower member: exposure Gini 0.857 vs "
    "0.929, and 42.9% of its slots go to long-tail (non-head) books vs only 10.0% for the "
    "two-tower. (b) Yet it narrows genres: mean recommended genre entropy is 0.811 bits against "
    "1.551 bits in readers' own histories, and the list is narrower than the history for 76.3% of "
    "warm readers. (c) Retraining item-item on its own served lists for 3 rounds concentrates it "
    "-- coverage 0.490 -> 0.412, Gini 0.773 -> 0.835, genre entropy 1.640 -> 1.184, clean hit "
    "0.252 -> 0.206 -- while the exploration slot ends round 3 at coverage 0.455, Gini 0.812, "
    "entropy 1.528 and clean hit 0.216: it partly restores reach and spread, it does not stop the "
    "drift. No single number here is 'the fairness' of the system: the same blend looks fairer "
    "than the two-tower on Gini and long-tail share yet narrower than its readers on genre, and "
    "each metric is one descriptive proxy over one population and one notion of exposure (books, "
    "not authors or readers). The loop must never let a simulated reader accept one of their own "
    "val or other held-out books: that would write the answer into the training log, so the refit "
    "model would recover it from its own leaked row and clean hit would RISE for the wrong reason "
    "-- measuring leakage, not recommendation quality."
)

print(f"val cohort: {len(cohort)} readers")
print(f"{'loop':17s} {'round':>5s} {'coverage':>9s} {'gini':>6s} {'genre H':>8s} {'clean hit':>10s}")
for row in loop_rows:
    print(f"{row['loop']:17s} {row['round']:5d} {row['coverage']:9.3f} {row['gini']:6.3f} "
          f"{row['genre_entropy']:8.3f} {row['clean_hit']:10.3f}")

assert [m.round for m in loop_plain] == [0, 1, 2, 3] and len(loop_explore) == 4
assert loop_plain[-1].coverage < loop_plain[0].coverage, "the plain loop narrows coverage"
assert loop_plain[-1].gini > loop_plain[0].gini, "...and concentrates exposure"
assert loop_explore[-1].coverage > loop_plain[-1].coverage, "exploration partly restores reach"

val cohort: 500 readers
loop              round  coverage   gini  genre H  clean hit
no mitigation         0     0.490  0.773    1.640      0.252
no mitigation         3     0.412  0.835    1.184      0.206
exploration slot      0     0.514  0.751    1.912      0.252
exploration slot      3     0.455  0.812    1.528      0.216


## Question 7 -- Build the Part-2 blend on `val`, then FREEZE it

Now measure the pinned Part-2 blend from Question 6 -- equal weights over item-item, two-tower,
semantic and lexical, pool 30 -- on `val` with `run_blended_scoreboard`, and record its hit@10 and
catalog coverage. Then run a **leave-one-path-out ablation** over its four members: drop one path,
re-blend the other three at the same pool with equal weights, and record the signed change in hit@10
and in coverage. Report the hybrid from Question 3 as a **separate single path** for comparison, not
as a blend member (it already contains the two-tower, so blending both would double-count it).

Read your numbers honestly. On this data the plan's measurement of this exact blend was about **hit
0.298 / coverage 0.383**: it **loses** on hit@10 to its own two-tower member (0.340) and to the hybrid
(0.362), and dropping either content path (semantic or lexical) *raises* hit@10. What it buys is
**reach**: about twice the two-tower's catalog coverage (0.177) and the hybrid's (0.192). This is a
**hit <-> coverage trade**, not a win.

Then freeze. Put the members, weights and pool into `FROZEN_PATHS`, `FROZEN_WEIGHTS` and
`FROZEN_POOL`, and record the frozen `val` hit@10 and coverage in the cell marked `# FREEZE`. You
freeze the blend *as specified* -- do not swap in the two-tower or drop a member to chase hit@10, and
never revisit this choice after Question 8. In three or four sentences: compare the frozen blend with
the best single path on **both** axes, say what the ablation shows, and say who would prefer each
recommender.

In [9]:
blend_val = run_blended_scoreboard(
    PART2_PATHS, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
    weights=PART2_WEIGHTS, cold_readers=cold_readers,
)                                       # split defaults to "val"
blend_val_hit = blend_val.hit_rate_at_k
blend_val_coverage = blend_val.catalog_coverage

ablation_rows = []
for dropped in ("item-item", "two-tower", "semantic", "lexical"):
    kept = {name: path for name, path in PART2_PATHS.items() if name != dropped}
    board = run_blended_scoreboard(
        kept, interactions_path, catalog_ids=catalog_ids, k=K, pool=POOL,
        weights={name: 1.0 for name in kept}, cold_readers=cold_readers,
    )
    ablation_rows.append({
        "dropped": dropped,
        "blend_hit@10": board.hit_rate_at_k,
        "delta_hit": board.hit_rate_at_k - blend_val_hit,               # signed
        "delta_coverage": board.catalog_coverage - blend_val_coverage,  # signed
    })
delta_hit_by_drop = {row["dropped"]: row["delta_hit"] for row in ablation_rows}

# The hybrid is a SEPARATE single path (it already contains the two-tower), never a blend member.
comparison = [
    {"recommender": "Part-2 blend", "hit@10": blend_val_hit, "coverage": blend_val_coverage},
    {"recommender": "two-tower", "hit@10": two_tower_hit, "coverage": two_tower_coverage},
    {"recommender": "hybrid", "hit@10": hybrid_hit, "coverage": hybrid_coverage},
]
blend_beats_best_single_on_hit = bool(blend_val_hit >= two_tower_hit)

print(f"Part-2 blend on val: hit@10 {blend_val_hit:.3f}   coverage {blend_val_coverage:.3f}")
print(f"{'dropped':10s} {'hit@10':>7s} {'d hit':>7s} {'d coverage':>11s}")
for row in ablation_rows:
    print(f"{row['dropped']:10s} {row['blend_hit@10']:7.3f} {row['delta_hit']:+7.3f} "
          f"{row['delta_coverage']:+11.3f}")
print(f"{'recommender':13s} {'hit@10':>7s} {'coverage':>9s}")
for row in comparison:
    print(f"{row['recommender']:13s} {row['hit@10']:7.3f} {row['coverage']:9.3f}")

assert 0.27 < blend_val_hit < 0.33, "the pinned blend's val hit@10 is about 0.298"
assert not blend_beats_best_single_on_hit, "the pinned blend LOSES to its two-tower on hit@10"
assert blend_val_hit < hybrid_hit, "...and to the hybrid"
assert blend_val_coverage > 1.8 * two_tower_coverage, "it buys about twice the two-tower's reach"
assert blend_val_coverage > 1.8 * hybrid_coverage, "...and about twice the hybrid's"
assert delta_hit_by_drop["semantic"] > 0 and delta_hit_by_drop["lexical"] > 0, (
    "dropping either content path RAISES hit@10"
)
assert delta_hit_by_drop["two-tower"] < 0 and delta_hit_by_drop["item-item"] < 0

Part-2 blend on val: hit@10 0.298   coverage 0.383
dropped     hit@10   d hit  d coverage
item-item    0.246  -0.052      -0.063
two-tower    0.198  -0.100      +0.052
semantic     0.310  +0.012      -0.019
lexical      0.312  +0.014      +0.035
recommender    hit@10  coverage
Part-2 blend    0.298     0.383
two-tower       0.340     0.177
hybrid          0.362     0.192


In [10]:
# FREEZE -- every Part-2 decision is settled here, on val. Nothing above this cell may change after
# Question 8 is run.
FROZEN_PATHS = dict(PART2_PATHS)       # the four members, exactly as pinned
FROZEN_WEIGHTS = dict(PART2_WEIGHTS)   # equal weights
FROZEN_POOL = POOL                     # 30
frozen_val_hit = blend_val_hit
frozen_val_coverage = blend_val_coverage

blend_reading = (
    "Against the best single path the frozen blend LOSES on hit@10 -- 0.298 vs the two-tower's "
    "0.340 and the hybrid's 0.362 -- but reaches about twice their catalog (coverage 0.383 vs "
    "0.177 and 0.192): a hit <-> coverage trade, not a win. The ablation shows the collaborative "
    "members carry the accuracy (dropping the two-tower costs 0.100 hit, item-item 0.052), while "
    "dropping either content path RAISES hit (semantic +0.012, lexical +0.014); semantic buys a "
    "little reach (coverage falls 0.019 without it) but lexical does not even do that here "
    "(coverage rises 0.035 without it), so not every member earns its place on either axis. "
    "Neither recommender dominates: a reader who wants the likeliest next read is better served "
    "by the hybrid or the two-tower, while a catalog owner, an author in the long tail, or a "
    "reader who wants discovery would prefer the blend's reach -- and we freeze it exactly as "
    "pinned rather than dropping members to chase hit@10."
)

print(f"FROZEN: members {list(FROZEN_PATHS)}, weights {FROZEN_WEIGHTS}, pool {FROZEN_POOL}")
print(f"frozen val hit@10 {frozen_val_hit:.3f}   coverage {frozen_val_coverage:.3f}")

assert list(FROZEN_PATHS) == ["item-item", "two-tower", "semantic", "lexical"]
assert set(FROZEN_WEIGHTS.values()) == {1.0} and FROZEN_POOL == 30

FROZEN: members ['item-item', 'two-tower', 'semantic', 'lexical'], weights {'item-item': 1.0, 'two-tower': 1.0, 'semantic': 1.0, 'lexical': 1.0}, pool 30
frozen val hit@10 0.298   coverage 0.383


## Question 8 -- Unseal `test`: one honest final score

The main log's `test` split has been sealed since Checkpoint A. Now -- and only now, with every
decision frozen at the end of Question 7 -- score the FROZEN blend on it **exactly once**: one call to
`run_blended_scoreboard` with `FROZEN_PATHS`, `FROZEN_WEIGHTS`, `FROZEN_POOL`, the same cold-reader
exclusion, and `split="test"`. Write that call **once**, at the top level of the cell -- not in a loop,
a function or a condition -- and run the cell once. Report the test hit@10, the test catalog coverage
and the number of readers scored.

For the exposure Gini, think before you compute: the blend's top-10 for a reader depends only on that
reader's **train** history, never on which held-out split is being graded. So the frozen blend's
exposure Gini over the warm population is the number you already measured in Question 6(a) --
unsealing `test` cannot change it. Report it alongside the test numbers.

Then write three or four sentences of honest interpretation:
- how do the `test` numbers compare with the frozen `val` numbers (about 0.298 / 0.383)?
- how do they compare with Checkpoint A's frozen Part-1 blend, which scored about **0.306 hit@10 /
  0.338 coverage on `test`** -- did Part 2's extra machinery buy accuracy, reach, or neither, *for
  this blend*? (Remember the two-tower and the hybrid you did not freeze.)
- why must you not go back now and freeze the two-tower instead, even though Question 7 says it has
  the higher `val` hit@10?

In [11]:
# The authorized one-shot unseal of the sealed main-log holdout. Run this cell ONCE; nothing in
# Questions 1-7 changes after reading it.
blend_test = run_blended_scoreboard(
    FROZEN_PATHS, interactions_path, catalog_ids=catalog_ids, k=K, pool=FROZEN_POOL,
    weights=FROZEN_WEIGHTS, cold_readers=cold_readers, split="test",
)
test_hit = blend_test.hit_rate_at_k
test_coverage = blend_test.catalog_coverage
test_readers = blend_test.readers

# The blend's top-10 depends only on TRAIN history, so its exposure Gini is the Question 6(a) value.
frozen_gini = blend_gini

final_report = {
    "val": (frozen_val_hit, frozen_val_coverage),
    "unsealed": (test_hit, test_coverage, test_readers),
    "exposure_gini": frozen_gini,
}

final_interpretation = (
    "On the unsealed holdout the frozen blend scores hit@10 0.286 and coverage 0.381 over 500 "
    "readers, close to its frozen val 0.298 / 0.383 over 500 readers -- a drop of about 0.012 "
    "hit, the size of difference a few hundred readers' sampling produces, so the val estimate "
    "was not an optimistic fluke. Against Checkpoint A's frozen Part-1 blend (about 0.306 hit / "
    "0.338 coverage on the same holdout) this Part-2 blend is about 0.02 LOWER on hit and about "
    "0.04 HIGHER on coverage: for THIS blend, Part 2's machinery bought reach, not accuracy -- "
    "the accuracy gains live in the two-tower and the hybrid we did not freeze, and its exposure "
    "Gini stays 0.857 because the served lists depend only on train history. Going back now to "
    "freeze the two-tower because its val hit is higher would be choosing AFTER looking at the "
    "holdout: the final number would then measure how well we picked against this split, turning "
    "it into a second validation set and leaving us no honest estimate at all."
)

print(f"frozen val:  hit@10 {frozen_val_hit:.3f}   coverage {frozen_val_coverage:.3f}   "
      f"({blend_val.readers} readers)")
print(f"unsealed:    hit@10 {test_hit:.3f}   coverage {test_coverage:.3f}   ({test_readers} readers)")
print(f"exposure Gini (carried from Question 6a): {frozen_gini:.3f}")

assert test_readers > 300, "the unsealed score aggregates over several hundred held-out readers"
assert 0.0 <= test_hit <= 1.0 and 0.0 < test_coverage <= 1.0
assert frozen_gini == blend_gini, "the label-free exposure Gini is unchanged by the unseal"

frozen val:  hit@10 0.298   coverage 0.383   (500 readers)
unsealed:    hit@10 0.286   coverage 0.381   (500 readers)
exposure Gini (carried from Question 6a): 0.857
